In [1]:
"""
Lab-4: Applied Statistical Modeling — Medical Insurance Cost Dataset
=====================================================================
Part 1: Exploratory Data Analysis & Hypothesis Testing
Part 2: Statistical Modeling (OLS) & Gauss-Markov Diagnostic Checks

Dataset: Medical Cost Personal Datasets (Kaggle - mirichoi0218/insurance)
https://www.kaggle.com/datasets/mirichoi0218/insurance

Run:
    python analysis.py

Outputs (saved to ./results/):
    - Descriptive statistics (csv)
    - Distribution / correlation / scatter plots (png)
    - Hypothesis test results (txt)
    - OLS regression summary (txt) + residual diagnostic plots (png)
"""

import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import jarque_bera, omni_normtest

sns.set_theme(style="whitegrid")
os.makedirs("results", exist_ok=True)

ALPHA = 0.05


In [2]:

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
df = pd.read_csv("data/insurance.csv")
print(f"Loaded {len(df)} rows, {df.shape[1]} columns")
print(df.head(), "\n")


Loaded 1338 rows, 7 columns
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520 



In [3]:

# ---------------------------------------------------------------------------
# PART 1a — Descriptive statistics
# ---------------------------------------------------------------------------
print("=" * 70)
print("PART 1a — DESCRIPTIVE STATISTICS")
print("=" * 70)

num_cols = ["age", "bmi", "children", "charges"]
desc = df[num_cols].describe().T
desc["skewness"] = df[num_cols].skew()
desc["kurtosis"] = df[num_cols].kurt()
desc["IQR"] = df[num_cols].quantile(0.75) - df[num_cols].quantile(0.25)
print(desc.round(3))
desc.round(3).to_csv("results/descriptive_stats.csv")

# ---------------------------------------------------------------------------
# PART 1b — Visual exploration
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, col in zip(axes.flat, num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="#4C72B0")
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.savefig("results/01_distributions.png", dpi=120)
plt.close()

plt.figure(figsize=(6, 5))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix (numeric features)")
plt.tight_layout()
plt.savefig("results/02_correlation_matrix.png", dpi=120)
plt.close()

plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="bmi", y="charges", hue="smoker", alpha=0.6)
plt.title("BMI vs Charges by Smoking Status")
plt.tight_layout()
plt.savefig("results/03_bmi_charges_smoker.png", dpi=120)
plt.close()

plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", alpha=0.6)
plt.title("Age vs Charges by Smoking Status")
plt.tight_layout()
plt.savefig("results/04_age_charges_smoker.png", dpi=120)
plt.close()

# ---------------------------------------------------------------------------
# PART 1c — Hypothesis Test 1: Smokers vs Non-Smokers (Charges)
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PART 1c — HYPOTHESIS TEST 1: Smokers vs Non-Smokers (Charges)")
print("=" * 70)
print("H0: mean(charges | smoker) = mean(charges | non-smoker)")
print("H1: mean(charges | smoker) != mean(charges | non-smoker)")

smoker_charges = df.loc[df.smoker == "yes", "charges"]
nonsmoker_charges = df.loc[df.smoker == "no", "charges"]

sh_smoker = stats.shapiro(smoker_charges.sample(min(500, len(smoker_charges)), random_state=1))
sh_nonsmoker = stats.shapiro(nonsmoker_charges.sample(min(500, len(nonsmoker_charges)), random_state=1))
print(f"\nShapiro-Wilk (smokers):     W={sh_smoker.statistic:.4f}, p={sh_smoker.pvalue:.4g}")
print(f"Shapiro-Wilk (non-smokers): W={sh_nonsmoker.statistic:.4f}, p={sh_nonsmoker.pvalue:.4g}")

levene = stats.levene(smoker_charges, nonsmoker_charges)
print(f"Levene's test (equal variance): stat={levene.statistic:.4f}, p={levene.pvalue:.4g}")

is_normal = (sh_smoker.pvalue > ALPHA) and (sh_nonsmoker.pvalue > ALPHA)
if is_normal:
    equal_var = levene.pvalue > ALPHA
    test_res = stats.ttest_ind(smoker_charges, nonsmoker_charges, equal_var=equal_var)
    test1_name = f"Two-sample t-test (equal_var={equal_var})"
else:
    test_res = stats.mannwhitneyu(smoker_charges, nonsmoker_charges, alternative="two-sided")
    test1_name = "Mann-Whitney U test"

decision1 = "Reject H0" if test_res.pvalue < ALPHA else "Fail to Reject H0"
print(f"\n{test1_name}: statistic={test_res.statistic:.4f}, p-value={test_res.pvalue:.4g}")
print(f"Decision at alpha=0.05: {decision1}")
print(f"Mean charges — smokers: {smoker_charges.mean():.2f} | non-smokers: {nonsmoker_charges.mean():.2f}")

# ---------------------------------------------------------------------------
# PART 1d — Hypothesis Test 2: One-Way ANOVA — Charges across Regions
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PART 1d — HYPOTHESIS TEST 2 (ANOVA): Charges across Regions")
print("=" * 70)
print("H0: mean charges are equal across all 4 regions")
print("H1: at least one region's mean charges differ")

groups = [df.loc[df.region == r, "charges"] for r in df.region.unique()]
anova = stats.f_oneway(*groups)
decision2 = "Reject H0" if anova.pvalue < ALPHA else "Fail to Reject H0"
print(f"F={anova.statistic:.4f}, p={anova.pvalue:.4g}")
print(f"Decision at alpha=0.05: {decision2}")
print(df.groupby("region")["charges"].mean().round(2))

with open("results/hypothesis_test_results.txt", "w") as f:
    f.write("HYPOTHESIS TEST 1 — Smoker vs Non-Smoker Charges\n")
    f.write(f"Test used: {test1_name}\n")
    f.write(f"statistic={test_res.statistic}\np-value={test_res.pvalue}\nDecision: {decision1}\n\n")
    f.write("HYPOTHESIS TEST 2 — ANOVA (charges across regions)\n")
    f.write(f"F={anova.statistic}\np-value={anova.pvalue}\nDecision: {decision2}\n")


PART 1a — DESCRIPTIVE STATISTICS
           count       mean        std       min       25%       50%  \
age       1338.0     39.207     14.050    18.000    27.000    39.000   
bmi       1338.0     30.663      6.098    15.960    26.296    30.400   
children  1338.0      1.095      1.205     0.000     0.000     1.000   
charges   1338.0  13270.422  12110.011  1121.874  4740.287  9382.033   

                75%        max  skewness  kurtosis        IQR  
age          51.000     64.000     0.056    -1.245     24.000  
bmi          34.694     53.130     0.284    -0.051      8.398  
children      2.000      5.000     0.938     0.202      2.000  
charges   16639.913  63770.428     1.516     1.606  11899.625  

PART 1c — HYPOTHESIS TEST 1: Smokers vs Non-Smokers (Charges)
H0: mean(charges | smoker) = mean(charges | non-smoker)
H1: mean(charges | smoker) != mean(charges | non-smoker)

Shapiro-Wilk (smokers):     W=0.9396, p=3.625e-09
Shapiro-Wilk (non-smokers): W=0.8787, p=2.415e-19
Levene's 

In [4]:

# ---------------------------------------------------------------------------
# PART 2a — OLS Regression
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PART 2a — MULTIPLE LINEAR REGRESSION (OLS)")
print("Model: charges ~ age + bmi + children + sex + smoker + region")
print("=" * 70)

model = smf.ols(
    "charges ~ age + bmi + children + C(sex) + C(smoker) + C(region)",
    data=df,
).fit()
print(model.summary())

with open("results/ols_summary.txt", "w") as f:
    f.write(str(model.summary()))

# ---------------------------------------------------------------------------
# PART 2b — Gauss-Markov diagnostic checks
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PART 2b — DIAGNOSTIC CHECKS")
print("=" * 70)

fitted = model.fittedvalues
resid = model.resid

# Residuals vs Fitted (linearity & homoscedasticity)
plt.figure(figsize=(7, 5))
plt.scatter(fitted, resid, alpha=0.4, s=15, color="#4C72B0")
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values")
plt.tight_layout()
plt.savefig("results/05_residuals_vs_fitted.png", dpi=120)
plt.close()

# Q-Q plot (normality of residuals)
plt.figure(figsize=(6, 6))
sm.qqplot(resid, line="45", fit=True)
plt.title("Q-Q Plot of Residuals")
plt.tight_layout()
plt.savefig("results/06_qq_plot.png", dpi=120)
plt.close()

jb_stat, jb_p, skew, kurt = jarque_bera(resid)
omni_stat, omni_p = omni_normtest(resid)
print(f"Jarque-Bera test: stat={jb_stat:.4f}, p={jb_p:.4g}")
print(f"Omnibus test:     stat={omni_stat:.4f}, p={omni_p:.4g}")

# Multicollinearity — VIF
X_design = model.model.exog
vif_data = pd.DataFrame({
    "feature": model.model.exog_names,
    "VIF": [variance_inflation_factor(X_design, i) for i in range(X_design.shape[1])],
})
print("\nVariance Inflation Factors:")
print(vif_data.round(3))
vif_data.round(3).to_csv("results/vif.csv", index=False)

with open("results/ols_summary.txt", "a") as f:
    f.write(f"\n\nJarque-Bera: stat={jb_stat:.4f}, p={jb_p:.4g}\n")
    f.write(f"Omnibus: stat={omni_stat:.4f}, p={omni_p:.4g}\n\n")
    f.write("Variance Inflation Factors:\n")
    f.write(vif_data.round(3).to_string(index=False))

print("\nAll results saved to ./results/")



PART 2a — MULTIPLE LINEAR REGRESSION (OLS)
Model: charges ~ age + bmi + children + sex + smoker + region
                            OLS Regression Results                            
Dep. Variable:                charges   R-squared:                       0.751
Model:                            OLS   Adj. R-squared:                  0.749
Method:                 Least Squares   F-statistic:                     500.8
Date:                Mon, 14 Sep 2026   Prob (F-statistic):               0.00
Time:                        16:15:38   Log-Likelihood:                -13548.
No. Observations:                1338   AIC:                         2.711e+04
Df Residuals:                    1329   BIC:                         2.716e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
-------------